# 02 — Bronze: land the raw gamelogs with Auto Loader

Ingest the raw gamelog JSON from the landing Volume into a single **VARIANT** Delta table
using **Auto Loader** in batch mode (`trigger(availableNow=True)`).

| Choice | Why (Well-Architected Framework) |
|--------|----------------------------------|
| `VARIANT` column | Keep the raw payload exactly as scraped — schema-on-read, no early data loss (**Reliability**) |
| Auto Loader + checkpoint | Exactly-once, incremental, idempotent re-runs (**Operational Excellence**) |
| `CLUSTER BY (file_batch_time)` | Liquid clustering for the silver reads that follow (**Performance**) |

Each landed file is a JSON **array** of game rows for one team-season, so one bronze row =
one file, and `data` is a VARIANT array we explode in silver.

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()
spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA",  "basketball_reference_waf")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/raw_data"

BRONZE_TABLE  = f"{UC_CATALOG}.{BRONZE_SCHEMA}.gamelog_raw"
SOURCE_PATH   = f"{VOLUME_PATH}/team_gamelog"
CHECKPOINT    = f"{VOLUME_PATH}/team_gamelog/_checkpoints/gamelog_raw"
SCHEMA_LOC    = f"{VOLUME_PATH}/team_gamelog/_schemas/gamelog_raw"
print("Bronze table:", BRONZE_TABLE)
print("Source path :", SOURCE_PATH)

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:141: UserWarning: Could not enable SetAllowOversizeProtos, please check the protobuf version.
  warnings.warn("Could not enable SetAllowOversizeProtos, please check the protobuf version.")


Bronze table: alexander_booth.basketball_reference_waf_bronze.gamelog_raw
Source path : /Volumes/alexander_booth/basketball_reference_waf_bronze/raw_data/team_gamelog


## Create the bronze table

In [2]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
        data                   VARIANT   COMMENT 'Raw gamelog rows for one team-season (JSON array), parsed to VARIANT',
        source_file            STRING    COMMENT 'Full path of the source file in the landing Volume',
        source_file_name       STRING    COMMENT 'File name, e.g. team=BOS.json',
        source_file_size       BIGINT    COMMENT 'Source file size in bytes',
        file_modification_time TIMESTAMP COMMENT 'Object-store last-modified time',
        file_batch_time        TIMESTAMP COMMENT 'Wall-clock time Auto Loader landed this row'
    )
    USING DELTA
    CLUSTER BY (file_batch_time)
    COMMENT 'Bronze - raw Basketball Reference team gamelogs, one row per source file. Re-runnable via Auto Loader checkpoints.'
""")
print("Bronze table ready.")

Bronze table ready.


## Run Auto Loader (batch)

`wholetext` reads each file as one string; `PARSE_JSON` turns it into a VARIANT array.

In [3]:
query = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "text")
        .option("wholetext", "true")
        .option("cloudFiles.schemaLocation", SCHEMA_LOC)
        .load(SOURCE_PATH)
        .selectExpr(
            "PARSE_JSON(value)                AS data",
            "_metadata.file_path              AS source_file",
            "_metadata.file_name              AS source_file_name",
            "_metadata.file_size              AS source_file_size",
            "_metadata.file_modification_time AS file_modification_time",
            "current_timestamp()              AS file_batch_time",
        )
        .writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT)
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
)
query.awaitTermination()
print("Auto Loader batch complete.")

Auto Loader batch complete.


## Verify

In [4]:
n = spark.table(BRONZE_TABLE).count()
print(f"Bronze rows (team-season files): {n}")

# Peek: how many game rows are inside one file's VARIANT array, and a sample row.
spark.sql(f"""
    SELECT source_file_name,
           variant_get(data, '$[0].team', 'string')          AS team,
           variant_get(data, '$[0].date', 'string')           AS first_game_date,
           variant_get(data, '$[0].opp_name_abbr', 'string')  AS first_opp
    FROM {BRONZE_TABLE}
    ORDER BY source_file_name
    LIMIT 5
""").show(truncate=False)

Bronze rows (team-season files): 90


+----------------+----+---------------+---------+
|source_file_name|team|first_game_date|first_opp|
+----------------+----+---------------+---------+
|team=ATL.json   |ATL |2022-10-19     |HOU      |
|team=ATL.json   |ATL |2024-10-23     |BRK      |
|team=ATL.json   |ATL |2023-10-25     |CHO      |
|team=BOS.json   |BOS |2024-10-22     |NYK      |
|team=BOS.json   |BOS |2023-10-25     |NYK      |
+----------------+----+---------------+---------+



Raw gamelogs are in bronze as VARIANT. **Next:** `03_silver_games_team_box` types them into
clean `games` and `team_game_box` tables and computes the Four Factors.